# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal 1: CTR (Click-Through Rate)

**Hypothesis:** Pages with low CTR are more likely to be declining in search performance.

**Method:** Bucketed pages into Zero / Low / Higher CTR groups, then compared the
distribution of `trend_direction` (down/stable/up/flat/new) across buckets.

**Observation:**
- Low CTR bucket: 59.3% down
- Higher CTR bucket: 44.1% down
- Zero CTR bucket: 49.7% down, but with a higher share of "new" pages (15.0%),
  which likely haven't had time to decline yet

**Verdict: MIXED**
The direction is correct (lower CTR → more decline), but the Zero-CTR bucket is
noisy because it mixes genuinely low-performing pages with newly published pages
that haven't accumulated a trend yet.

In [3]:
!pwd
!ls

/content
sample_data


In [4]:
!git clone https://github.com/QasimNaveed-stack/flyrank-ml-internship.git
%cd flyrank-ml-internship
!ls

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 128, done.
remote: Counting objects: 100% (128/128), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 128 (delta 42), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (128/128), 1.86 MiB | 11.46 MiB/s, done.
Resolving deltas: 100% (42/42), done.
/content/flyrank-ml-internship
AGENTS.md  DATA_USE.md	LICENSE    README.md	     SETUP.md	 work
CLAUDE.md  docs		notebooks  requirements.txt  skills
data	   GUIDE.md	outputs    scripts	     submission


In [6]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [7]:
list(df.columns)

['content_id',
 'client_id',
 'search_volume',
 'competition',
 'competition_level',
 'cpc',
 'content_type',
 'main_intent',
 'word_count',
 'char_count',
 'provider_used',
 'model_used',
 'impressions_90d',
 'clicks_90d',
 'pageviews_90d',
 'sessions_90d',
 'users_90d',
 'engaged_sessions_90d',
 'ai_sessions_90d',
 'scroll_events_90d',
 'days_with_impressions',
 'days_with_sessions',
 'impressions_last_30d',
 'clicks_last_30d',
 'sessions_last_30d',
 'impressions_prev_30d',
 'clicks_prev_30d',
 'sessions_prev_30d',
 'content_age_days',
 'age_tier',
 'age_tier_order',
 'days_since_last_update',
 'freshness_tier',
 'word_count_tier',
 'char_count_tier',
 'ctr',
 'avg_position',
 'engagement_rate',
 'scroll_rate',
 'ai_traffic_pct',
 'impression_tier',
 'position_tier',
 'trend_direction',
 'trend_pct']

In [9]:
def ctr_bucket(x):
    if x == 0:
        return 'Zero'
    elif x < 1.0:
        return 'Low'
    else:
        return 'Higher'

df['ctr_bucket'] = df['ctr'].apply(ctr_bucket)

bucket_table = df.groupby('ctr_bucket')['trend_direction'].value_counts(normalize=True).unstack()
bucket_table

trend_direction,down,flat,new,stable,up
ctr_bucket,,,,,
Higher,0.440609,0.029842,0.076068,0.232300,0.221182
Low,0.593342,0.000531,0.007892,0.260694,0.137542
Zero,0.496670,0.082728,0.150394,0.123675,0.146533


In [10]:
def position_bucket(x):
    if x == 0:
        return 'No_Data'
    elif x <= 10:
        return 'Top_10'
    elif x <= 20:
        return 'Position_11_20'
    else:
        return 'Below_20'

df['position_bucket'] = df['avg_position'].apply(position_bucket)

bucket_table_2 = df.groupby('position_bucket')['trend_direction'].value_counts(normalize=True).unstack()
bucket_table_2

trend_direction,down,flat,new,stable,up
position_bucket,,,,,
Below_20,0.528165,0.014873,0.039466,0.201897,0.215599
No_Data,0.006639,0.035685,0.957676,NaN,NaN
Position_11_20,0.609515,0.016362,0.026949,0.214354,0.132820
Top_10,0.563121,0.066472,0.042286,0.206347,0.121775


## Signal 2: Average Search Position

**Hypothesis:** Pages with worse average position are more likely to be declining.

**Observation:**
- Top_10: 56.3% down
- Position_11_20: 61.0% down
- Below_20: 52.8% down
- No_Data (95.8% "new" pages): excluded from trend interpretation

**Verdict: MIXED**
No clean monotonic relationship between position and decline — the worst-position
bucket (Below_20) does not show the highest decline rate. Position alone is not
a reliable standalone signal here.

In [11]:
df[['ctr', 'sessions_90d', 'scroll_rate', 'impressions_90d']].describe()

,ctr,sessions_90d,scroll_rate,impressions_90d
count,30000.000000,30000.000000,29875.000000,30000.000000
mean,0.510733,37.066633,18.212921,5200.366300
std,3.279162,107.069131,29.472768,16838.019547
min,0.000000,1.000000,0.000000,1.000000
25%,0.000000,2.000000,0.000000,81.000000
50%,0.070000,7.000000,5.000000,731.000000
75%,0.290000,27.000000,23.530000,3615.250000
max,100.000000,4345.000000,300.000000,517715.000000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [12]:
# Fill missing scroll_rate with 0 (no scroll data recorded = no engagement observed)
df['scroll_rate'] = df['scroll_rate'].fillna(0)

# Step 1: Individual low-signal flags (based on median thresholds)
df['low_ctr_flag'] = (df['ctr'] < 0.07).astype(int)
df['low_sessions_flag'] = (df['sessions_90d'] < 7).astype(int)
df['low_scroll_flag'] = (df['scroll_rate'] < 5.0).astype(int)

# Step 2: Visibility check — only pages that actually get impressions are worth refreshing
df['visible_flag'] = (df['impressions_90d'] >= 731).astype(int)

# Step 3: Score = sum of weak signals, but only counted if the page is visible
df['weak_signal_count'] = df['low_ctr_flag'] + df['low_sessions_flag'] + df['low_scroll_flag']
df['score'] = df['weak_signal_count'] * df['visible_flag'] * df['impressions_90d']

# Step 4: Reason codes
def get_reason_code(row):
    codes = []
    if row['low_ctr_flag']:
        codes.append('LOW_CTR')
    if row['low_sessions_flag']:
        codes.append('LOW_SESSIONS')
    if row['low_scroll_flag']:
        codes.append('LOW_SCROLL')
    if row['weak_signal_count'] == 3 and row['visible_flag'] == 1:
        codes.append('HIGH_PRIORITY')
    return '|'.join(codes) if codes else 'NONE'

df['reason_code'] = df.apply(get_reason_code, axis=1)

# Step 5: Rank
df_ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
df_ranked['rank'] = df_ranked.index + 1

# Step 6: Write CSV
import os
os.makedirs('work/outputs', exist_ok=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print("Saved! Shape:", df_ranked.shape)
df_ranked[['rank', 'content_id', 'score', 'reason_code', 'ctr', 'sessions_90d', 'scroll_rate', 'impressions_90d']].head(10)

Saved! Shape: (30000, 54)


,rank,content_id,score,reason_code,ctr,sessions_90d,scroll_rate,impressions_90d
0,1,content_c8e9d6ab9013,626034,LOW_CTR|LOW_SESSIONS|LOW_SCROLL|HIGH_PRIORITY,0.00,6,0.00,208678
1,2,content_36ff89c8214e,590194,LOW_CTR|LOW_SCROLL,0.05,238,4.48,295097
2,3,content_8451fc6f034d,544288,LOW_CTR|LOW_SCROLL,0.03,99,2.26,272144
3,4,content_aaef01a50def,517109,LOW_SCROLL,0.25,1527,4.77,517109
4,5,content_ff94c9b6b411,457132,LOW_CTR|LOW_SCROLL,0.04,87,1.12,228566
5,6,content_a023517539fe,428094,LOW_CTR|LOW_SCROLL,0.01,147,3.61,214047
6,7,content_1a9e894be2e2,416180,LOW_SCROLL,0.23,1140,4.97,416180
7,8,content_2c2606c5d176,347399,LOW_SCROLL,0.53,2146,3.28,347399
8,9,content_0e70a832cb7a,346900,LOW_CTR|LOW_SCROLL,0.04,117,2.31,173450
9,10,content_db5989a78dd3,345111,LOW_SCROLL,0.21,647,3.72,345111


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [13]:
top20 = df_ranked.head(20)[['rank', 'content_id', 'score', 'reason_code', 'ctr', 'sessions_90d', 'scroll_rate', 'impressions_90d', 'content_type', 'main_intent']]
top20

,rank,content_id,score,reason_code,ctr,sessions_90d,scroll_rate,impressions_90d,content_type,main_intent
0,1,content_c8e9d6ab9013,626034,LOW_CTR|LOW_SESSIONS|LOW_SCROLL|HIGH_PRIORITY,0.00,6,0.00,208678,keyword article,informational
1,2,content_36ff89c8214e,590194,LOW_CTR|LOW_SCROLL,0.05,238,4.48,295097,keyword article,informational
2,3,content_8451fc6f034d,544288,LOW_CTR|LOW_SCROLL,0.03,99,2.26,272144,keyword article,informational
3,4,content_aaef01a50def,517109,LOW_SCROLL,0.25,1527,4.77,517109,keyword article,informational
4,5,content_ff94c9b6b411,457132,LOW_CTR|LOW_SCROLL,0.04,87,1.12,228566,keyword article,informational
5,6,content_a023517539fe,428094,LOW_CTR|LOW_SCROLL,0.01,147,3.61,214047,keyword article,informational
6,7,content_1a9e894be2e2,416180,LOW_SCROLL,0.23,1140,4.97,416180,keyword article,transactional
7,8,content_2c2606c5d176,347399,LOW_SCROLL,0.53,2146,3.28,347399,keyword article,commercial
8,9,content_0e70a832cb7a,346900,LOW_CTR|LOW_SCROLL,0.04,117,2.31,173450,keyword article,informational
9,10,content_db5989a78dd3,345111,LOW_SCROLL,0.21,647,3.72,345111,keyword article,commercial


## Top-20 Manual Review

**Pattern observed:** Two distinct groups appear in the top 20.

**Group A — Genuinely weak pages** (ranks 1,2,3,5,6,9,11,15,17,18,20):
Both LOW_CTR and LOW_SCROLL flags present. CTR values are near-zero (0.00–0.06),
well below the dataset median (0.07). These are strong, confident refresh candidates.
- **Action:** refresh_and_review
- **Reason:** genuinely low engagement across two independent signals
- **Confidence:** high
- **What would make this wrong:** if the page targets a highly niche/technical query
  where low CTR is expected regardless of quality (would need query-level context
  to confirm)

**Group B — High-volume pages with only one weak signal** (ranks 4,7,8,10,12,13,16,19):
CTR here is actually healthy (0.21–0.89, at or above median). Only LOW_SCROLL is
flagged. These rank highly only because impressions_90d is very large, which
dominates the score formula.
- **Action:** review_scroll_only (lower priority than Group A)
- **Reason:** LOW_SCROLL
- **Confidence:** medium — CTR being healthy suggests the page is being found and
  clicked; the scroll issue may be a content-layout problem, not a "declining"
  content problem
- **What would make this wrong:** if these pages are long-form and scroll_rate
  naturally reads low for legitimately good pages (short answer boxes, etc.) —
  would need to check word_count for these rows to confirm

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [14]:
weak_picks = df_ranked[df_ranked['reason_code'] == 'LOW_SCROLL'].head(10)
weak_picks[['content_id', 'word_count', 'scroll_rate', 'ctr', 'impressions_90d']]

,content_id,word_count,scroll_rate,ctr,impressions_90d
3,content_aaef01a50def,NaN,4.77,0.25,517109
6,content_1a9e894be2e2,NaN,4.97,0.23,416180
7,content_2c2606c5d176,NaN,3.28,0.53,347399
9,content_db5989a78dd3,2682.0,3.72,0.21,345111
11,content_44e481c8f55b,3040.0,2.78,0.65,312694
12,content_cb112fce36be,2761.0,2.95,0.16,309910
15,content_89e84d699e9e,3480.0,1.08,0.89,275226
18,content_aa4baf490b43,2715.0,1.90,0.50,256290
20,content_008fb02c46cb,3338.0,1.47,0.26,236803
25,content_73c54f78c06a,2658.0,3.97,0.10,213963


## Weak Picks + Leakage Check

**Weak picks identified:** Ranks with only the LOW_SCROLL reason code (e.g. ranks
4, 7, 8, 10, 12, 13, 16, 19) are questionable top-20 entries. Checking `word_count`
for these rows shows they are long-form articles (2,600–3,480 words where the
value is present; several rows have missing word_count). Long articles naturally
show lower scroll_rate because fewer readers scroll to the bottom of a long page —
this is a content-length artifact, not necessarily a sign of declining performance.
Their CTR (0.16–0.89) is at or above the dataset median (0.07), meaning search
visibility and click interest are healthy.

**Why this happened:** The current score formula treats every weak signal equally
and multiplies by raw impressions_90d, so a single LOW_SCROLL flag on a
high-traffic, long-form page can outrank pages with multiple genuine weak signals
but lower traffic. A future improvement would be to normalize scroll_rate by
content length or word_count tier before flagging it as "low."

**Leakage check:**
- No future-window columns used (`impressions_last_30d`/`prev_30d` were available
  but not used in the score — only the full `_90d` aggregates were used, consistent
  with the March 2026 decision-time window from the Week 3 data contract)
- `trend_direction` and `trend_pct` were used only for signal *validation*
  (Section 1 analysis), never as inputs to the score itself — consistent with the
  label-trap warning in `flyrank-data/SKILL.md`
- No client-provided product flags or business labels were used
- Score is built entirely from `ctr`, `sessions_90d`, `scroll_rate`, and
  `impressions_90d` — all observed, pre-decision metrics

In [15]:
!ls work/outputs/

baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.